In [ ]:
!pip install Sastrawi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.4 MB/s eta 0:00:00


In [ ]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 9.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import re
import emoji
import pickle
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, SpatialDropout1D, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

# Import Library Sastrawi untuk Bahasa Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# 1. LOAD DATA
from google.colab import files
print("Silakan upload file 'INA_TweetsPPKM_Labeled_Pure.csv'...")
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# 2. BACA DATASET
try:
    df = pd.read_csv(file_name, sep='\t')
    if df.shape[1] < 2: raise ValueError
except:
    try:
        df = pd.read_csv(file_name, sep=',')
    except:
        df = pd.read_csv(file_name, sep=None, engine='python')

# 3. FILTER HANYA POSITIF & NEGATIF
label_col = next((col for col in df.columns if 'sentiment' in col.lower() or 'sentimen' in col.lower()), None)
text_col = next((col for col in df.columns if 'tweet' in col.lower() or 'text' in col.lower()), None)

# Buang Netral (1)
df = df[df[label_col] != 1].copy()
print(f"Sisa Data (Positif & Negatif): {len(df)}")

# 4. PREPROCESSING LANJUTAN (SASTRAWI)
# Siapkan Stemmer (Pencari kata dasar)
factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Siapkan Stopword (Penghapus kata sambung)
stop_factory = StopWordRemoverFactory()
stopword = stop_factory.create_stop_word_remover()

def clean_text_advanced(text):
    if not isinstance(text, str): return ""
    # 1. Hapus Emoji & URL
    text = emoji.replace_emoji(text, replace='')
    text = re.sub(r"http\S+|www\S+|@\S+|#\S+", "", text)
    # 2. Hapus Angka & Tanda Baca
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = text.lower().strip()

    # 3. Stopword Removal (Hapus 'yang', 'dan', dll)
    text = stopword.remove(text)

    # 4. Stemming (Lama prosesnya, tapi bikin akurasi naik)
    # Ubah "memperpanjang" -> "panjang"
    text = stemmer.stem(text)

    return text

print("\nSedang membersihkan teks dengan Sastrawi... (Mungkin agak lama 1-2 menit)")
df['text_clean'] = df[text_col].apply(clean_text_advanced)
df = df[df['text_clean'].str.len() > 0]

# 5. ENCODING & SPLIT
le = LabelEncoder()
df['label_encoded'] = le.fit_transform(df[label_col]) # Positif=0, Negatif=1

X_train, X_test, y_train, y_test = train_test_split(df['text_clean'], df['label_encoded'], test_size=0.2, random_state=42, stratify=df['label_encoded'])

# 6. CLASS WEIGHT
class_weights_array = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(zip(np.unique(y_train), class_weights_array))

# 7. TOKENISASI
vocab_size = 8000 # Kurangi vocab size biar model lebih fokus
max_length = 80   # Tweet biasanya pendek, 80 kata cukup
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=max_length, padding='post', truncating='post')
test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=max_length, padding='post', truncating='post')

# 8. BANGUN MODEL (ANTI-OVERFITTING)
model = Sequential([
    Embedding(vocab_size, 100, input_length=max_length),
    SpatialDropout1D(0.3), # Dropout di embedding

    # LSTM Layer (Unit dikurangi jadi 64 biar gak terlalu kompleks/menghapal)
    Bidirectional(LSTM(64, dropout=0.3, recurrent_dropout=0.3, return_sequences=False)),

    # Regularization L2 (Hukuman biar model gak curang menghapal)
    Dense(32, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.5), # Matikan 50% neuron secara acak saat training

    Dense(2, activation='softmax')
])

# Optimizer dengan Learning Rate agak kecil biar belajarnya teliti
opt = tf.keras.optimizers.Adam(learning_rate=0.0005)

model.compile(loss='sparse_categorical_crossentropy', optimizer=opt, metrics=['accuracy'])

# 9. TRAINING (DENGAN CALLBACKS PINTAR)
# ReduceLROnPlateau: Kalau akurasi mentok, akan diturunkan kecepatan belajarnya
# secara otomatis
callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001, verbose=1)
]

print("\n=== MULAI TRAINING VERSI TUNED ===")
history = model.fit(
    train_pad, y_train,
    epochs=20, # Tambah epoch karena ada EarlyStopping
    batch_size=32,
    validation_split=0.1,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

# 10. EVALUASI & SIMPAN
y_pred = np.argmax(model.predict(test_pad), axis=-1)
print("\n=== HASIL AKHIR ===")
# Ingat: 0=Positif, 1=Negatif (karena label asli 2 berubah jadi 1 setelah encode)
print(classification_report(y_test, y_pred, target_names=["Positif", "Negatif"]))

# Simpan
model.save('model_sentimen_lstm.h5')
with open('tokenizer.pkl', 'wb') as f: pickle.dump(tokenizer, f)
with open('label_encoder.pkl', 'wb') as f: pickle.dump(le, f)

Silakan upload file 'INA_TweetsPPKM_Labeled_Pure.csv'...


Saving INA_TweetsPPKM_Labeled_Pure.csv to INA_TweetsPPKM_Labeled_Pure.csv
Sisa Data (Positif & Negatif): 5938

Sedang membersihkan teks dengan Sastrawi... (Mungkin agak lama 1-2 menit)

=== MULAI TRAINING VERSI TUNED ===
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


132/132 ━━━━━━━━━━━━━━━━━━━━ 34s 196ms/step - accuracy: 0.5205 - loss: 0.7414 - val_accuracy: 0.6546 - val_loss: 0.6615 - learning_rate: 5.0000e-04
Epoch 2/20
132/132 ━━━━━━━━━━━━━━━━━━━━ 25s 189ms/step - accuracy: 0.7569 - loss: 0.5674 - val_accuracy: 0.8358 - val_loss: 0.4464 - learning_rate: 5.0000e-04
Epoch 3/20
132/132 ━━━━━━━━━━━━━━━━━━━━ 41s 188ms/step - accuracy: 0.8798 - loss: 0.3549 - val_accuracy: 0.8465 - val_loss: 0.4345 - learning_rate: 5.0000e-04
Epoch 4/20
132/132 ━━━━━━━━━━━━━━━━━━━━ 41s 188ms/step - accuracy: 0.9222 - loss: 0.2641 - val_accuracy: 0.8486 - val_loss: 0.4240 - learning_rate: 5.0000e-04
Epoch 5/20
132/132 ━━━━━━━━━━━━━━━━━━━━ 41s 187ms/step - accuracy: 0.9473 - loss: 0.1923 - val_accuracy: 0.8465 - val_loss: 0.4569 - learning_rate: 5.0000e-04
Epoch 6/20
132/132 ━━━━━━━━━━━━━━━━━━━━ 25s 189ms/step - accuracy: 0.9627 - loss: 0.1367 - val_accuracy: 0.8614 - val_loss: 0.4580 - learning_rate: 5.0000e-04
Epoch 7/20
132/132 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step - a


=== HASIL AKHIR ===
              precision    recall  f1-score   support

     Positif       0.73      0.78      0.76       383
     Negatif       0.89      0.86      0.88       789

    accuracy                           0.84      1172
   macro avg       0.81      0.82      0.82      1172
weighted avg       0.84      0.84      0.84      1172

